# 🏗️ Notebook 1: Netflix — Requirements & Architecture

## 🛠️ Setup

```bash
cd 06-system-designs/netflix
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## What we're designing

A global video streaming service. Think Netflix: hundreds of millions of users, petabytes
of video, low startup latency worldwide, personalized recommendations.

If you've never thought about streaming before, here's the one-sentence version:
**most of the work is moving video bytes close to the user, not computing anything clever
at request time.** The design is dominated by that fact.

### Functional requirements (the happy path)
- Upload & publish a video (admin side) — transcoded into multiple resolutions.
- Browse and search the catalog.
- **Stream** a video with good startup time and minimal rebuffering.
- Personalized **recommendations** ("Because you watched…").
- Track **playback position** so users resume where they left off.

### Non-functional
- **Read-heavy**: 99% of traffic is *watching*, not *uploading*. Optimize for reads.
- **Low startup**: <2s from click to first frame.
- **High availability**: 99.99%+ for playback. A dead catalog is bad; a dead player is worse.
- **Global**: users in Asia should not pull bytes from US-East.


## Back-of-envelope — let's actually compute it

Instead of guessing, let's put the numbers in Python so you can tweak them.

In [ ]:
# Back-of-envelope calculator. Change the inputs and re-run.
# Everything below is derived — no number is typed in twice.

users              = 200_000_000
peak_concurrent_pct = 0.15           # 15% of subscribers watching at peak
avg_bitrate_mbps    = 3.0            # blended SD/HD/4K stream

concurrent       = users * peak_concurrent_pct
peak_egress_tbps = concurrent * avg_bitrate_mbps / 1_000_000   # Mbps -> Tbps

print(f"Peak concurrent viewers: {concurrent:>15,.0f}")
print(f"Peak egress:             {peak_egress_tbps:>15,.1f} Tbps")

# A single large cloud region tops out near ~10 Tbps of egress.
dc_cap_tbps = 10
print(f"Data centers needed (no CDN): {peak_egress_tbps / dc_cap_tbps:.0f}")
print("\nThat number is the entire argument for a CDN. It is not an optimisation —")
print("it is the only way the bytes physically leave the building.")


### Storage — derive it from bitrate, not from the master file

The tempting shortcut is `titles x master_size x renditions`. **That is wrong**, and wrong in
both directions: a 240p rendition is nowhere near the size of a ProRes master, and a 4K
rendition of a 3-hour film dwarfs the "average". Encoded size is set by exactly one thing:

```
bytes = bitrate (bits/s) x duration (s) / 8
```

So we sum the *ladder* — every rendition's bitrate — and multiply by duration. Then we multiply
again by the number of **codec families** we ship, which is the part people forget: Netflix
encodes the same ladder in H.264 (universal compatibility), VP9, and AV1 (better compression on
devices that support it). Three ladders, not one.

In [ ]:
catalog_titles  = 20_000
avg_duration_h  = 1.5                       # blended films + episodes
audio_mbps      = 0.5                       # a few language tracks + descriptive audio
codec_families  = 3                         # H.264 (compat) + VP9 + AV1
mezzanine_mbps  = 100                       # the ProRes-ish master we keep forever

# The bitrate ladder actually shipped to players.
LADDER_MBPS = {"240p": 0.4, "360p": 0.8, "480p": 1.4, "720p": 2.8, "1080p": 5.0, "4k": 15.0}

duration_s  = avg_duration_h * 3600
ladder_mbps = sum(LADDER_MBPS.values())

def gb(mbps: float, seconds: float) -> float:
    """Megabits/sec x seconds -> gigabytes.  Mb/s * s = Mb;  /8 -> MB;  /1000 -> GB."""
    return mbps * seconds / 8 / 1000

video_gb_per_title = gb(ladder_mbps, duration_s) * codec_families
audio_gb_per_title = gb(audio_mbps, duration_s)
mezz_gb_per_title  = gb(mezzanine_mbps, duration_s)
per_title_gb       = video_gb_per_title + audio_gb_per_title + mezz_gb_per_title

print(f"Bitrate ladder total: {ladder_mbps:.1f} Mbps across {len(LADDER_MBPS)} renditions")
print(f"\nPer title ({avg_duration_h}h):")
print(f"  {'encoded video (' + str(codec_families) + ' codecs)':<32}{video_gb_per_title:>7.1f} GB")
print(f"  {'audio tracks':<32}{audio_gb_per_title:>7.1f} GB")
print(f"  {'mezzanine master (kept forever)':<32}{mezz_gb_per_title:>7.1f} GB")
print(f"  {'TOTAL':<32}{per_title_gb:>7.1f} GB")

catalog_pb = catalog_titles * per_title_gb / 1e6
print(f"\nWhole catalog ({catalog_titles:,} titles): {catalog_pb:.2f} PB")
print(f"  ...of which servable renditions: "
      f"{catalog_titles * (video_gb_per_title + audio_gb_per_title) / 1e6:.2f} PB")

# Sanity-check the shortcut we warned about.
naive_tb = catalog_titles * 5 * len(LADDER_MBPS) / 1024
print(f"\nThe naive 'titles x 5GB master x 6 renditions' shortcut gives {naive_tb:,.0f} TB")
print(f"— {catalog_pb * 1000 / naive_tb:.1f}x too small, because it charges 240p the price of a")
print("master and never counts the master itself or the extra codec ladders.")

print("\nWhat this number is for: it tells you the ORIGIN is an object-store problem")
print("(petabytes, cheap, cold) while the EDGE is a working-set problem — a POP only")
print("caches the popular few TB, which is the next cell's point.")


## Bad → Best: where do the bytes come from?

Let's compare three architectures. The "bad" one is a naive first draft; each step fixes
the previous pain point.

In [ ]:
# --- v1 (BAD): origin serves all video ---
# One region, one data center, streams video directly to users.
# Users in Tokyo pull bytes from us-east-1. Latency is awful, egress is capped.
v1_problems = [
    f"Egress bottleneck (~{dc_cap_tbps} Tbps cap vs {peak_egress_tbps:.0f} Tbps demand)",
    "Trans-ocean latency (~150ms RTT)",
    "One viral episode can DoS the origin",
    "Bandwidth cost: you pay cloud egress for every single byte",
]
for p in v1_problems: print(" -", p)

# --- v2 (BETTER): add a CDN in front of the origin ---
# Origin stores video; CDN caches popular chunks at the edge (POPs).
# Cache-hit rate is the whole game. Let's compute origin load for different hit rates.
def origin_tbps(total_tbps: float, hit_rate: float) -> float:
    return total_tbps * (1 - hit_rate)

print("\nWith CDN in front of origin:")
for h in (0.50, 0.90, 0.95, 0.99):
    load = origin_tbps(peak_egress_tbps, h)
    fits = "fits one region" if load <= dc_cap_tbps else "STILL OVER BUDGET"
    print(f"  hit rate {h:.0%} -> origin needs {load:5.1f} Tbps   ({fits})")

# --- v3 (BEST): CDN + multi-region origin + ISP-embedded caches ---
# Netflix OpenConnect puts caching appliances INSIDE ISPs.
# Result: the popular 99% of bytes never leave the ISP network -> near-zero backbone cost.
#
# The honest cost of v3, which the diagram never shows:
#   - You must physically ship, power and remotely operate hardware in other people's
#     data centers, and negotiate with every ISP individually.
#   - Cache fill is its own traffic problem: pre-positioning new releases to thousands of
#     appliances during off-peak hours is a scheduled, bandwidth-shaped job.
#   - A bug in the appliance image is a fleet-wide outage you cannot hotfix from your laptop.
# This only pays off at Netflix's scale. Below that, rent a commercial CDN.


### Why cache hit rate is so high for video

Because catalog popularity follows a Zipf / power-law distribution: a small number of
titles account for most views. Let's see that.

In [ ]:
import random
from collections import Counter

random.seed(0)
N_TITLES = 20_000
N_REQUESTS = 100_000

# Zipf: popularity proportional to 1/rank^s
s = 1.1
weights = [1 / ((i + 1) ** s) for i in range(N_TITLES)]
total = sum(weights)
probs = [w / total for w in weights]

requests = random.choices(range(N_TITLES), weights=probs, k=N_REQUESTS)
counts = Counter(requests)

top500 = {t for t, _ in counts.most_common(500)}
hits = sum(1 for r in requests if r in top500)
print(f"Top-500 titles account for {hits/N_REQUESTS:.1%} of requests")
print("-> A cache holding just 500 titles serves most traffic.")


## High-level architecture

```
  [User]
    | https
    v
  +--------------------+
  |   Edge CDN (POPs)  |<-- 99% of bytes served here
  +--------+-----------+
           | cache miss / metadata
           v
  +--------------------+   +-----------------+
  |   API Gateway      |-->|  Auth, Rate lim |
  +--------+-----------+   +-----------------+
           |
   +-------+-------+---------------+--------------+
   v       v       v               v              v
 Catalog  User   Playback       Recommendation  Watch-history
 Service  Svc    Svc (manifests)   Svc           Svc
   |       |       |                |              |
   v       v       v                v              v
 MySQL  MySQL   Object Store    ML feature   Cassandra
                (S3)            store + models
                   ^
                   | write after encoding
  +----------------+------------------+
  |     Encoding / transcoding        |  (batch jobs, ffmpeg workers)
  |  mezzanine -> HLS/DASH x bitrates |
  +-----------------------------------+
```

### Key responsibilities
- **CDN**: primary *bytes* delivery. Netflix has its own (OpenConnect). A from-scratch
  design might use Cloudflare / Akamai / CloudFront.
- **Manifest**: a tiny playlist (HLS `.m3u8` or DASH MPD) telling the player which chunks
  to fetch at which bitrate.
- **ABR (Adaptive Bit Rate)**: player picks quality based on measured bandwidth, re-deciding
  every chunk. We simulate this in notebook 3.
- **Recommendations** are *precomputed* offline; serving is just a cache lookup.

### Why split "control plane" from "data plane"?
- Control plane (catalog, auth, recs lookup) = small JSON, CPU-bound, needs consistency.
- Data plane (video bytes) = huge, cacheable, needs bandwidth.
These have opposite scaling profiles - keeping them separate lets each scale on its own.


## Takeaways
1. **Do the back-of-envelope first.** It immediately tells you "a CDN is mandatory."
2. **Bad → Best is about cache hit rate.** CDN + popularity skew → origin load shrinks 20x.
3. **Split control plane from data plane.** Never stream video through your JSON API.
